# CRAFT-GC — Image-Level Fairness Pipeline

**Required:** Runtime → **T4 GPU** → Save → **Restart session** → Run Cell 1 first.

**Secrets:** Add `HF_TOKEN` in Colab Secrets (HuggingFace Read token).

| Cell | Task | Expected time |
|------|------|---------------|
| 1 | Setup + clone latest code | ~5 min |
| 2 | Stage A embeddings (optional) | ~2 min |
| 3 | **Smoke test** (8 images) | ~3–5 min |
| 4 | **Full run** (600 images) | **2–4 hours** |
| 5 | Download | ~1 min |

⚠️ If any cell finishes in **under 1 minute** (except Cell 2/5), it **failed** — read the red error text.

In [ ]:
# CELL 1 — Setup (run after GPU restart)
import os, sys, shutil, subprocess
from getpass import getpass

os.chdir('/content')
REPO = 'https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git'
PROJECT = '/content/Research'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'open-clip-torch', 'diffusers', 'transformers',
    'accelerate', 'pandas', 'matplotlib', 'seaborn', 'tqdm', 'huggingface_hub'], check=True)

from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login(token=getpass('HF token: '))

if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', REPO, PROJECT], cwd='/content', check=True)
subprocess.run(['git', '-C', PROJECT, 'pull', 'origin', 'main'], check=False)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT], check=True)

import torch, craft_gc
if not torch.cuda.is_available():
    raise RuntimeError('NO GPU. Set Runtime → T4 GPU → Save → Restart → re-run Cell 1.')

commit = subprocess.check_output(['git', '-C', PROJECT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
script = open('scripts/run_pilot_images.py').read()
if '--model_id' not in script:
    raise RuntimeError('OLD scripts cloned. Push latest code to GitHub and re-run Cell 1.')

print('='*50)
print('SETUP OK | git', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('craft_gc:', craft_gc.__version__)
print('CWD:', os.getcwd())
print('='*50)

In [ ]:
# CELL 2 — Stage A (optional, ~2 min)
import os
os.chdir('/content/Research')
!python scripts/evaluate_embeddings.py --device cuda

In [ ]:
# CELL 3 — SMOKE TEST (8 images, ~3–5 min) — MUST PASS before Cell 4
import os, subprocess, sys
from pathlib import Path

PROJECT = '/content/Research'
os.chdir(PROJECT)

def run_script(args, label):
    print('\n' + '='*60)
    print(label)
    print('='*60)
    r = subprocess.run([sys.executable] + args, cwd=PROJECT)
    if r.returncode != 0:
        raise RuntimeError(f'{label} FAILED (exit code {r.returncode}). Read errors above.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — enable T4 and restart.')

run_script([
    'scripts/run_pilot_images.py',
    '--device', 'cuda',
    '--limit', '2',
    '--seeds', '42',
    '--out', 'results/smoke_test',
    '--model_id', 'SG161222/Realistic_Vision_V5.1_noVAE',
    '--vae_id', 'stabilityai/sd-vae-ft-mse',
], 'SMOKE: 2 prompts × 4 methods × 1 seed = 8 images')

pngs = list(Path('results/smoke_test').rglob('*.png'))
print(f'Smoke PNG count: {len(pngs)}')
if len(pngs) != 8:
    raise RuntimeError(f'Expected 8 PNGs, found {len(pngs)}. Do NOT run Cell 4.')
print('SMOKE TEST PASSED — safe to run Cell 4 (full 600-image run)')

In [ ]:
# CELL 4 — FULL RUN (600 images, 2–4 HOURS) — do not close tab
import os, subprocess, sys, json
from pathlib import Path

PROJECT = '/content/Research'
os.chdir(PROJECT)

def run_script(args, label):
    print('\n' + '='*60)
    print(label)
    print('='*60)
    r = subprocess.run([sys.executable] + args, cwd=PROJECT)
    if r.returncode != 0:
        raise RuntimeError(f'{label} FAILED (exit {r.returncode})')

EXPECTED = 50 * 3 * 4  # prompts × seeds × methods
print(f'Generating {EXPECTED} images — expect 2–4 hours on T4')

run_script([
    'scripts/run_pilot_images.py',
    '--device', 'cuda',
    '--limit', '50',
    '--seeds', '42', '123', '456',
    '--methods', 'base', 'prompt_aug', 'fairimagen', 'craftgc',
    '--model_id', 'SG161222/Realistic_Vision_V5.1_noVAE',
    '--vae_id', 'stabilityai/sd-vae-ft-mse',
], f'GENERATION ({EXPECTED} images)')

pngs = list(Path('results/pilot_images').rglob('*.png'))
manifest = json.loads(Path('results/pilot_images/manifest.json').read_text())
print(f'PNG files: {len(pngs)} | Manifest entries: {len(manifest)}')
if len(pngs) != EXPECTED or len(manifest) != EXPECTED:
    raise RuntimeError(f'Expected {EXPECTED} images, got {len(pngs)} PNGs / {len(manifest)} manifest rows')

run_script([
    'scripts/evaluate_images.py',
    '--device', 'cuda',
    '--manifest', 'results/pilot_images/manifest.json',
    '--output', 'results/image_eval_v2.csv',
], 'IMAGE EVALUATION (CLIP zero-shot CoFS)')

run_script(['scripts/merge_paper_results.py'], 'MERGE RESULTS')
run_script(['scripts/plot_figures.py'], 'PLOT FIGURES')
run_script(['scripts/make_qualitative_grid.py'], 'QUALITATIVE GRID')

print('\n' + '='*60)
print('FULL PIPELINE COMPLETE')
print('Run Cell 5 to download results/')
print('='*60)

In [ ]:
# CELL 5 — Download
import os, shutil
from pathlib import Path
from google.colab import files

os.chdir('/content/Research')
n = len(list(Path('results/pilot_images').rglob('*.png')))
print(f'Packaging {n} images...')
if n < 600:
    print(f'WARNING: only {n}/600 images — Cell 4 may not have finished correctly.')

shutil.make_archive('/content/craft_gc_results', 'zip', 'results')
shutil.make_archive('/content/craft_gc_figures', 'zip', 'craft-gc-paper/figures')
files.download('/content/craft_gc_results.zip')
files.download('/content/craft_gc_figures.zip')
print('Done. Unzip into ~/Desktop/Research/ on your PC.')